<a href="https://colab.research.google.com/github/acchetan/AIML_Course/blob/main/Chetan_Python_Taskmanager_with_user_authentication_in_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task Manager with User Authentication

Course-end Project 2
Description
Problem Statement:

In today’s world, individuals often need to keep track of various tasks in a structured way. You are tasked with building a Task Manager that allows users to manage their tasks. The system should include user authentication, meaning each user has to log in with a username and password. Once logged in, users can create, view, update, and delete their tasks. Each user’s tasks should be stored separately, and only the authenticated user can access their tasks.

Objectives:

1. Design and implement a user authentication system (login and registration)

2. Create a task management system that allows users to:

     a. Add, view, mark as completed, and delete tasks

3. Use file handling to store user credentials and tasks persistently

4. Create an interactive menu-driven interface to manage tasks




**Solution written by Chetan A C  starting on 28th May  and completed on 30 May after revising the basics of python and learning additional aspects needed for the assignment**

In [ ]:
!pip install -q streamlit pyngrok

import os
ngrok_authtokken=input("Enter the auth token for nGrok. If you don't have one, register in nGrok and get the token")


In [ ]:
%%writefile db.py
import pandas as pd
import sqlite3
import hashlib
from datetime import date

# Global variable for DB file (will be temporarily overridden for testing)
DB_file="task_manager.db"

# Global variable for test connection - will be set by unit tests
_test_conn = None

def get_db_connection():
    """Connect to the database and return a connection object."""
    global _test_conn
    if _test_conn: # If a test connection is set, use it
        return _test_conn
    # Otherwise, connect to the regular file DB
    conn = sqlite3.connect(DB_file)
    conn.row_factory = sqlite3.Row #allows to access columns by their name as in db.
    return conn

def init_db():
    """Creates the users table if it doesn't exist yet."""
    conn = get_db_connection()
    # Create a cursor object to execute SQL commands
    cursor = conn.cursor()
    # Create a simple tasks table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user TEXT NOT NULL,
            password TEXT NOT NULL,
            status TEXT DEFAULT 'active'
        )
    ''')
    # Commit changes (needed for in-memory DB to persist table creation within a connection)
    conn.commit()
    #create a tasks table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user TEXT NOT NULL,
            title TEXT NOT NULL,
            status TEXT DEFAULT 'Pending',
            description TEXT DEFAULT "",
            due_date TEXT,
            active TEXT DEFAULT 'active'
        )
    ''')
    conn.commit()
    # Do NOT close connection if it's the test connection, as it will be reused
    if conn != _test_conn:
        conn.close()

def add_task(user, title,**task):
    """Inserts a new task into the database."""
    if task.get("description"):
        description = task["description"]
    else:
        description = ""
    if task.get("status"):
        Status = task["status"]
    else:
        Status = "Pending"
    if task.get("due_date"):
        due_date = task["due_date"]
    else:
        due_date = date.today()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO tasks (user, title, due_date, description, status) VALUES (?, ?, ?, ?, ?)",
        (user, title, due_date, description, Status)
    )
    conn.commit()
    conn.close()

def get_all_tasks(user):
    """Fetches all tasks for a given user and returns them as a list of dictionaries."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM tasks WHERE user = ? AND active ='active' ", (user,))
    rows = cursor.fetchall()
    conn.close()
    return [dict(row) for row in rows]

def update_task_status(task_id, new_status):
    """Updates the status of a specific task."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE tasks SET status = ? WHERE id = ?",
        (new_status, task_id)
    )
    conn.commit()
    conn.close()

def update_task_title(task_id, new_title):
    """Updates the title of a specific task."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE tasks SET title = ? WHERE id = ?",
        (new_title, task_id)
    )
    conn.commit()
    conn.close()

def update_task_description(task_id, new_description):
    """Updates the description of a specific description."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE tasks SET description = ? WHERE id = ?",
        (new_description, task_id)
    )
    conn.commit()
    conn.close()

def update_task_duedate(task_id, new_duedate):
    """Updates the description of a specific description."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE tasks SET due_date = ? WHERE id = ?",
        (new_duedate, task_id)
    )
    conn.commit()
    conn.close()

def delete_task(task_id):
    """marks a task as inactive by it's id"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE tasks SET active='inactive' WHERE id = ?", (task_id,))
    conn.commit()
    conn.close()

def fetch_users():
    """Fetches active usernames in the user table"""
    conn = get_db_connection()
    # Create a cursor object to execute SQL commands
    cursor = conn.cursor()
    cursor.execute('''SELECT user FROM users WHERE status = 'active' ''')
    rows = cursor.fetchall()
    if conn != _test_conn:
        conn.close() # Close connection after fetching
    # Convert SQLite rows into standard Python dictionaries and extract usernames
    users_data =[dict(row) for row in rows]
    user_list = [u['user'] for u in users_data]
    return user_list

def add_users(user, password):
    """Inserts a new user into the database."""
    if not user or not user.strip():
        return "User is blank, please enter a user name along with password"
    if not password or not password.strip():
        return "Password cannot be empty"

    hashed_pwd = hash_password(password)
    if hashed_pwd == "FALSE": # Should ideally not happen if password.strip() is checked
        return "Password hashing failed"

    conn = get_db_connection()
    cursor = conn.cursor()
    # Check if user already exists using fetch_users (which uses current DB_file)
    if user in fetch_users(): # fetch_users now returns list of usernames
        if conn != _test_conn:
            conn.close()
        return "User already exists, Please try another user name"
    else:
        cursor.execute(
            "INSERT INTO users (user, password) VALUES (?, ?)",
            (user, hashed_pwd) # Insert hashed password
        )
        conn.commit()
        if conn != _test_conn:
            conn.close()
        return "User added"

def hash_password(password):
    """Hashes a given password using SHA256."""
    if password and password.strip(): # Corrected to call strip()
        return hashlib.sha256(password.encode()).hexdigest()
    return "FALSE" # Return "FALSE" for empty/blank passwords


def authenticate_user(user, password):
    """Authenticates a user by checking their credentials in the database."""
    if not user or not user.strip():
        return "User is blank, please enter a user name along with password"
    if not password or not password.strip():
        return "Password cannot be empty"

    hashed_pwd = hash_password(password)
    if hashed_pwd == "FALSE":
        return "Password cannot be empty"

    # Check if user exists first without opening the main connection for password check
    if user not in fetch_users():
        return "User does not exist, please enter a valid user name or register new user"

    # If user exists, then get the connection to check password
    conn = get_db_connection()
    try:
        cursor = conn.cursor()
        # Check if user exists and password matches
        cursor.execute("SELECT password FROM users WHERE status = 'active' AND user =?", (user,))
        user_record = cursor.fetchone() # Get the record once

        if user_record:
            stored_password_hash = user_record[0]
            return stored_password_hash == hashed_pwd
        else:
            return "Wrong password, try again with correct password"
    finally:
        # Close connection only if it's not the test connection
        if conn != _test_conn:
            conn.close()


In [ ]:
# @title
import unittest
import sqlite3
import hashlib
import os
import db # Import the db module

# Temporarily store the original DB_file path
ORIGINAL_DB_FILE = db.DB_file

class TestUserManagement(unittest.TestCase):

    def setUp(self):
        """Set up an in-memory database for each test."""
        global ORIGINAL_DB_FILE # Declare ORIGINAL_DB_FILE as global to modify it

        # Create a single in-memory connection for the current test
        db._test_conn = sqlite3.connect(':memory:')
        db._test_conn.row_factory = sqlite3.Row

        # Initialize the users table using the test connection
        db.init_db()

    def tearDown(self):
        """Clean up after each test."""
        if db._test_conn:
            db._test_conn.close()
            db._test_conn = None # Clear the test connection
        db.DB_file = ORIGINAL_DB_FILE # Restore original DB_file path

    def test_get_db_connection(self):
        """Test if a database connection can be established (should return _test_conn)."""
        conn = db.get_db_connection()
        self.assertIsNotNone(conn)
        self.assertEqual(conn.row_factory, sqlite3.Row)
        self.assertEqual(conn, db._test_conn) # Ensure it's the test connection
        # No need to close conn here as it's _test_conn and closed in tearDown

    def test_init_users(self):
        """Test if the users table is created correctly."""
        conn = db.get_db_connection() # This will return _test_conn
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(users)")
        columns = cursor.fetchall()
        column_names = [col['name'] for col in columns]
        self.assertIn('id', column_names)
        self.assertIn('user', column_names)
        self.assertIn('password', column_names)
        self.assertIn('status', column_names)
        # No need to close conn here as it's _test_conn and closed in tearDown

    def test_hash_password(self):
        """Test password hashing functionality."""
        password = "testpassword"
        hashed = db.hash_password(password)
        self.assertIsInstance(hashed, str)
        self.assertEqual(len(hashed), 64) # SHA256 produces 64-char hex string
        self.assertNotEqual(hashed, password) # Hashed password should not be plain text

        # Test with empty/blank passwords
        self.assertEqual(db.hash_password(""), "FALSE")
        self.assertEqual(db.hash_password("   "), "FALSE")
        self.assertEqual(db.hash_password(None), "FALSE")

    def test_add_users(self):
        """Test adding new users and handling existing users."""
        # Add a new user
        result = db.add_users("testuser", "securepass")
        self.assertEqual(result, "User added")
        self.assertIn("testuser", db.fetch_users())

        # Try to add the same user again
        result = db.add_users("testuser", "anotherpass")
        self.assertEqual(result, "User already exists, Please try another user name")

        # Test with empty/blank user/password
        self.assertEqual(db.add_users("", "pass"), "User is blank, please enter a user name along with password")
        self.assertEqual(db.add_users("user", ""), "Password cannot be empty")
        self.assertEqual(db.add_users("   ", "pass"), "User is blank, please enter a user name along with password")
        self.assertEqual(db.add_users("user", "   "), "Password cannot be empty")

    def test_fetch_users(self):
        """Test fetching active users."""
        self.assertEqual(db.fetch_users(), []) # Should be empty initially

        db.add_users("user1", "pass1")
        db.add_users("user2", "pass2")
        users = db.fetch_users()
        self.assertIn("user1", users)
        self.assertIn("user2", users)
        self.assertEqual(len(users), 2)

    def test_authenticate_user(self):
        """Test user authentication process."""
        db.add_users("authuser", "authpass")

        # Correct credentials
        self.assertTrue(db.authenticate_user("authuser", "authpass"))

        # Incorrect password
        self.assertFalse(db.authenticate_user("authuser", "wrongpass"))

        # Non-existent user
        self.assertEqual(db.authenticate_user("nonexistent", "anypass"), "User does not exist, please enter a valid user name or register new user")

        # Empty/blank credentials
        self.assertEqual(db.authenticate_user("", "pass"), "User is blank, please enter a user name along with password")
        self.assertEqual(db.authenticate_user("user", ""), "Password cannot be empty")
        self.assertEqual(db.authenticate_user("   ", "pass"), "User is blank, please enter a user name along with password")
        self.assertEqual(db.authenticate_user("user", "   "), "Password cannot be empty")

# Run the tests
if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


In [ ]:
import os
import db

# Ensure a clean database state for main operations
if os.path.exists(db.DB_file):
    os.remove(db.DB_file)
    print(f"Removed existing database file: {db.DB_file}")

db.init_db()
print("Database initialized successfully for the Streamlit application.")


In [ ]:
%%writefile app.py
import streamlit as st
import db # Import the db module
import pandas as pd
from datetime import datetime
import json
import os

import pandas as pd
import sqlite3
import hashlib



# Set page configuration
st.set_page_config(page_title="Task Manager", page_icon="📝", layout="centered")

# --- INITIALIZE SESSION STATE ---
if "screen" not in st.session_state:
    st.session_state.screen = "landing"  # Options: 'landing', 'login', 'dashboard', 'register'

if "username" not in st.session_state:
    st.session_state.username = ""  #blank user name

if "tasks" not in st.session_state:
    st.session_state.tasks = [] # Initialize as empty, will load from DB for the user


# --- SCREEN 1: LANDING PAGE ---
if st.session_state.screen == "landing":
    st.write("")  # Spacer
    st.write("")

    # Main Title Box
    st.title("📝 Chetan's Task Manager App")

    st.write("")
    # Login Button Channeling
    col1, col2, col3 = st.columns([1, 1, 1])
    with col2:
        if st.button(
            "Login", use_container_width=True, type="primary"
        ):
            st.session_state.screen = "login"
            st.rerun()
    st.write("")
    #Register new user
    col1, col2, col3 = st.columns([1, 1, 1])
    with col2:
        if st.button(
            "Register new user", use_container_width=True, type="primary"
        ):
            st.session_state.screen = "register"
            st.rerun()
    st.write("")

# --- SCREEN 2: LOGIN PAGE ---
elif st.session_state.screen == "login":
    st.markdown("### Login")

    with st.form("login_form"):
        username = st.text_input("Username").strip()
        password = st.text_input("Password", type="password").strip()
        submit_button = st.form_submit_button("Submit")

        if submit_button:
            if username and password:
                auth_result = db.authenticate_user(username, password) # Use the authentication function
                if auth_result is True:
                    st.session_state.username = username
                    st.session_state.screen = "dashboard"
                    st.session_state.tasks = db.get_all_tasks(username) # Load user's tasks
                    st.rerun()
                else:
                    st.error(str(auth_result)) # Display error message from authenticate_user
            else:
                st.error("Please enter both a username and password.")

    if st.button("← Back"):
        st.session_state.screen = "landing"
        st.rerun()


# --- SCREEN 3: DASHBOARD (TASKS) ---
elif st.session_state.screen == "dashboard":
    # Welcome banner

    header_col1, header_col2 = st.columns([4, 1])

    with header_col1:
        st.subheader(f"Welcome, {st.session_state.username}! 👋")

    with header_col2:
        # Aligning the logout button right by using container width or a clean button block
        if st.button("Logout", use_container_width=True, key="top_logout"):
            st.session_state.screen = "landing"
            st.session_state.username = ""
            st.rerun()

    st.write("---")

    st.markdown("### Your Tasks")
    st.session_state.tasks = db.get_all_tasks(st.session_state.username)

    # --- SECTION 1: Add New Task ---
    st.subheader("Add a New Task")
    with st.form(key="add_task_form", clear_on_submit=True):
        col1, col2, col3 = st.columns([2,3,1])
        with col1:
            new_task_name = st.text_input("What needs to be done?")
        with col2:
            new_task_desc = st.text_input("Description")
        with col3:
            new_task_due = st.date_input("Due Date", min_value=datetime.today())

        submit_button = st.form_submit_button(label="Add Task")

        if submit_button and new_task_name.strip():
            db.add_task(st.session_state.username, new_task_name.strip(), due_date=str(new_task_due), description = new_task_desc.strip())
            st.session_state.tasks = db.get_all_tasks(st.session_state.username) # Reload tasks
            st.success(f"Task '{new_task_name}' added!")
            st.rerun()
        elif submit_button:
            st.error("Task name cannot be empty!") # Re-typed this line and surrounding block to fix potential SyntaxError

    # --- SECTION 2: Interactive Task Table ---
    if st.session_state.tasks:
        # Convert list of dicts to Dataframe for st.data_editor
        df = pd.DataFrame(st.session_state.tasks)
        df['due_date'] = pd.to_datetime(df['due_date'])

        # Render interactive grid allowing status changes and edits
        edited_df = st.data_editor(
            df,
            column_config={
                "id": None,  # Hide the ID column from the user
                "user": None, # Hide user column
                "title": st.column_config.TextColumn("Task Name", required=True),
                "description": st.column_config.TextColumn("Description"),
                "status": st.column_config.SelectboxColumn(
                    "Status",
                    options=["Pending", "In Progress", "Completed"],
                    required=True,
                ),
                "due_date": st.column_config.DateColumn("Due Date", required=True),
                "active":None
            },
            hide_index=True,
            use_container_width=True,
            num_rows="dynamic",
            key="task_data_editor"
        )

        # If the user changed anyting, sync back to database
        if not edited_df.equals(df):
            for _, row in edited_df.iterrows():
                original_task = df[df['id'] == row['id']].iloc[0]
                if original_task['status'] != row['status']:
                    db.update_task_status(row['id'], row['status'])
                if original_task['title'] != row['title']:
                    db.update_task_title(row['id'], row['title'])
                if original_task['description'] != row['description']:
                    db.update_task_description(row['id'], row['description'])
                if original_task['due_date'] != row['due_date']:
                      db.update_task_duedate(row['id'], row['due_date'].strftime('%Y-%m-%d')) # Convert Timestamp to string
            st.session_state.tasks = db.get_all_tasks(st.session_state.username) # Reload tasks
            st.rerun()
    else:
        st.info("No tasks yet. Add one above!")

    # --- SECTION 3: Quick Action / Delete ---
    with st.sidebar:
      st.sidebar.subheader("Quick Actions")

      if st.session_state.tasks:
          task_options = {t["title"]: t["id"] for t in st.session_state.tasks}
          task_to_delete = st.sidebar.selectbox("Select a task to delete:", options=list(task_options.keys()))

          if st.sidebar.button("🗑️ Delete Task", type="primary"):
              target_id = task_options[task_to_delete]
              db.delete_task(target_id)
              st.session_state.tasks = db.get_all_tasks(st.session_state.username) # Reload tasks
              st.rerun()
      else:
          st.sidebar.info("No tasks to delete.")




# --- SCREEN 4: Register new user ---
elif st.session_state.screen == "register":
    st.markdown("### Register New User")

    with st.form("New User Form"):
        username = st.text_input("Username").strip()
        password = st.text_input("Password", type="password").strip()
        submit_button = st.form_submit_button("Register")

        if submit_button:
            if username and password:
                add_result = db.add_users(username, password) # Use the add_users function - FIX: Changed to db.add_users
                if add_result == "User added":
                    st.success(f"User '{username}' registered successfully! You can now log in.")
                    st.session_state.screen = "login"
                    st.rerun()
                else:
                    st.error(str(add_result)) # Display error message from add_users
            else:
                st.error("Please enter both a username and password.")

    if st.button("← Back"):
        st.session_state.screen = "landing"
        st.rerun()

In [ ]:
import urllib
print("Your LocalTunnel Password/Endpoint IP is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

In [ ]:
# Terminate any previous Streamlit process
!kill $(lsof -t -i:8501)

# Terminate any existing ngrok tunnels
from pyngrok import ngrok
ngrok.kill()

# 1. Start Streamlit in the background
!streamlit run app.py &>/content/logs.txt &

# 2. Open the network tunnel
NGROK_AUTH_TOKEN = ngrok_authtokken
ngrok.set_auth_token(NGROK_AUTH_TOKEN) #Authenticate Ngrok


public_url = ngrok.connect(8501) #Open a tunnel on port 8501
print("👉 Click this link to open your app:", public_url.public_url)


### Streamlit Application Logs

In [ ]:
!cat /content/logs.txt